Imports

In [2]:

import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from sentence_transformers import SentenceTransformer


c:\Users\kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Dataset

In [5]:
DATASET_PATH = "../dataset.csv"
df = pd.read_csv(DATASET_PATH)

print("Dataset Loaded Successfully.")
print("Total Samples:", len(df))
print("Unique Intents:", df["intent"].nunique())

Dataset Loaded Successfully.
Total Samples: 6539
Unique Intents: 27


Label Encoding

In [6]:
label_encoder = LabelEncoder()
df["intent_encoded"] = label_encoder.fit_transform(df["intent"])

print("Label encoding completed.")


##New Addition
print("Model Labels:")
print(label_encoder.classes_)

Label encoding completed.
Model Labels:
['cancel_order' 'change_order' 'change_shipping_address'
 'check_cancellation_fee' 'check_invoice' 'check_payment_methods'
 'check_refund_policy' 'complaint' 'contact_customer_service'
 'contact_human_agent' 'create_account' 'delete_account'
 'delivery_options' 'delivery_period' 'edit_account' 'get_invoice'
 'get_refund' 'newsletter_subscription' 'payment_issue' 'place_order'
 'recover_password' 'registration_problems' 'review'
 'set_up_shipping_address' 'switch_account' 'track_order' 'track_refund']


Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df["utterance"],
    df["intent_encoded"],
    test_size=0.2,
    random_state=42,
    stratify=df["intent_encoded"]
)


Load Embedding Model

In [8]:
print("Loading SentenceTransformer model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")


Loading SentenceTransformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 647.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


Generate Embeddings

In [ ]:
print("Generating embeddings...")

X_train_embeddings = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_test_embeddings = embedder.encode(X_test.tolist(), show_progress_bar=True)

print("Embeddings generated.")


Generating embeddings...


Batches: 100%|██████████| 41/41 [00:01<00:00, 30.76it/s]

Embeddings generated.


Train Logistic Regression

In [10]:

model = LogisticRegression(max_iter=1000)
model.fit(X_train_embeddings, y_train)

print("Model training completed.")

Model training completed.


Evaluation

In [11]:
y_pred = model.predict(X_test_embeddings)
accuracy = accuracy_score(y_test, y_pred)

print("\n==========================================")
print("Model Accuracy:", round(accuracy * 100, 2), "%")
print("==========================================\n")

print("Classification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_
))



Model Accuracy: 99.62 %

Classification Report:

                          precision    recall  f1-score   support

            cancel_order       0.98      1.00      0.99        49
            change_order       1.00      0.98      0.99        46
 change_shipping_address       1.00      1.00      1.00        45
  check_cancellation_fee       1.00      1.00      1.00        49
           check_invoice       1.00      1.00      1.00        52
   check_payment_methods       1.00      1.00      1.00        49
     check_refund_policy       1.00      1.00      1.00        48
               complaint       1.00      0.98      0.99        50
contact_customer_service       1.00      1.00      1.00        51
     contact_human_agent       1.00      1.00      1.00        45
          create_account       1.00      0.98      0.99        47
          delete_account       1.00      0.98      0.99        47
        delivery_options       1.00      1.00      1.00        47
         delivery_period 

Save Model

In [12]:
with open("intent_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("\nModel files saved successfully.")
print("Embedding-based training pipeline complete.")


Model files saved successfully.
Embedding-based training pipeline complete.
